# E-Commerce Customer Analytics & Product Association System

**Author:** Machine Learning & Analytics Team  
**Dataset:** [UCI Online Retail Dataset](https://archive.ics.uci.edu/dataset/352/online+retail)  
**Algorithms:** 
1. **K-Means Clustering** (RFM Customer Segmentation)
2. **Apriori Association Rule Mining** (Market Basket Analysis)
3. **Random Forest Classification** (Customer Spending Tier Classification)

---
### Workflow Architecture:
* **Step 1:** Environment Setup & Library Imports
* **Step 2:** Robust Dataset Ingestion & Initial Inspection
* **Step 3:** Rigorous Data Cleaning, Validation & Revenue Calculation
* **Step 4:** Exploratory Data Analysis (EDA) & Business KPI Dashboards
* **Step 5:** RFM Feature Engineering & K-Means Clustering (Elbow & Silhouette Analysis)
* **Step 6:** Apriori Market Basket Analysis & Association Rules Generation
* **Step 7:** Leak-Free Random Forest Classification & Model Serialization


## Step 1: Import Libraries & Configure Random State

In [ ]:
import os
import sys
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from mlxtend.frequent_patterns import apriori, association_rules

# Settings and reproducibility
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("All required libraries imported successfully. Reproducible random state configured.")


## Step 2: Ingest & Inspect Dataset

We load the UCI Online Retail dataset using a robust search strategy that works regardless of whether the working directory is the project root or the `notebooks/` directory.


In [ ]:
# Locate dataset with robust relative path resolution
candidate_paths = [
    os.path.join("..", "dataset", "online_retail_cleaned.parquet"),
    os.path.join("dataset", "online_retail_cleaned.parquet"),
    os.path.join("..", "dataset", "online_retail.xlsx"),
    os.path.join("dataset", "online_retail.xlsx"),
    os.path.join("..", "dataset", "Online Retail.xlsx"),
    os.path.join("dataset", "Online Retail.xlsx"),
]

dataset_path = None
for p in candidate_paths:
    if os.path.exists(p):
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError(
        "Online Retail dataset not found! Please place 'online_retail.xlsx' inside the 'dataset/' directory."
    )

print(f"Loading dataset from: {dataset_path}")
if dataset_path.endswith('.parquet'):
    df_raw = pd.read_parquet(dataset_path)
else:
    df_raw = pd.read_excel(dataset_path)

print(f"Raw Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns\n")
display(df_raw.head())


In [ ]:
# Summary statistics and structure
print("--- Column Data Types & Non-Null Counts ---")
print(df_raw.info())

print("\n--- Missing Values Count & Percentage ---")
missing_df = pd.DataFrame({
    'Missing Count': df_raw.isnull().sum(),
    'Percentage (%)': (df_raw.isnull().sum() / len(df_raw)) * 100
})
display(missing_df)

print(f"\nDuplicate Records: {df_raw.duplicated().sum():,}")

print("\n--- Numerical Feature Distributions ---")
display(df_raw.describe())


## Step 3: Systematic Data Cleaning & Preprocessing

### Cleaning Strategy:
1. **Preserve Raw Data:** Keep `df_raw` untouched and produce a new `df_clean` DataFrame.
2. **Missing CustomerID:** Drop rows where `CustomerID` is NaN because customer-level analytics require an identifiable customer ID.
3. **Cancellations:** Invoices starting with the letter **'C'** indicate cancellations or credit notes. We exclude these from purchase behavior analysis.
4. **Non-Positive Quantities & Unit Prices:** Invalid transactions where `Quantity <= 0` or `UnitPrice <= 0` (adjustments, damages, errors) are filtered out.
5. **Duplicate Transactions:** Deduplicate identical item rows within the same order.
6. **Data Type Standardization:** Convert `InvoiceDate` to datetime, `CustomerID` to clean string integer (e.g. '17850'), and `StockCode`/`Description` to clean strings.
7. **Revenue Calculation:** Compute item line revenue: $\text{Revenue} = \text{Quantity} \times \text{UnitPrice}$.


In [ ]:
# Execute systematic data cleaning
total_raw = len(df_raw)

# 1. Handle missing CustomerID
df_with_cust = df_raw.dropna(subset=['CustomerID']).copy()
dropped_cust = total_raw - len(df_with_cust)

# 2. Filter out cancellations (InvoiceNo starting with 'C')
df_valid_inv = df_with_cust[~df_with_cust['InvoiceNo'].astype(str).str.startswith('C')].copy()
dropped_canc = len(df_with_cust) - len(df_valid_inv)

# 3. Filter positive quantity and unit price
df_pos = df_valid_inv[(df_valid_inv['Quantity'] > 0) & (df_valid_inv['UnitPrice'] > 0)].copy()
dropped_neg = len(df_valid_inv) - len(df_pos)

# 4. Remove duplicate rows
df_dedup = df_pos.drop_duplicates().copy()
dropped_dups = len(df_pos) - len(df_dedup)

# 5. Type standardization and feature creation
df_clean = df_dedup.copy()
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)
df_clean['StockCode'] = df_clean['StockCode'].astype(str).str.strip()
df_clean['Description'] = df_clean['Description'].astype(str).str.strip()
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str).str.strip()
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Audit Report
cleaning_audit = pd.DataFrame([
    {"Step": "Raw Transactions Ingested", "Records Retained": total_raw, "Records Removed": 0, "Reason": "Initial input"},
    {"Step": "Filter Missing CustomerID", "Records Retained": len(df_with_cust), "Records Removed": dropped_cust, "Reason": "Customer ID missing"},
    {"Step": "Filter Cancelled Invoices ('C')", "Records Retained": len(df_valid_inv), "Records Removed": dropped_canc, "Reason": "Cancellations/returns"},
    {"Step": "Filter Non-Positive Qty/Price", "Records Retained": len(df_pos), "Records Removed": dropped_neg, "Reason": "Zero or negative amounts"},
    {"Step": "Remove Duplicate Rows", "Records Retained": len(df_clean), "Records Removed": dropped_dups, "Reason": "Exact redundant lines"}
])

print("=== Data Cleaning Audit Summary ===")
display(cleaning_audit)
print(f"\nFinal Cleaned Dataset: {len(df_clean):,} valid transaction records ({len(df_clean)/total_raw*100:.1f}% retained)")
print(f"Unique Customers: {df_clean['CustomerID'].nunique():,}")
print(f"Date Range: {df_clean['InvoiceDate'].min().strftime('%Y-%m-%d')} to {df_clean['InvoiceDate'].max().strftime('%Y-%m-%d')}")
print(f"Total Net Sales Revenue: £{df_clean['Revenue'].sum():,.2f}")


## Step 4: Exploratory Data Analysis (EDA) & Key Metrics

Here we analyze high-level business indicators, monthly revenue patterns, top-performing products, and geographic distribution.


In [ ]:
# Core Business Metrics
total_revenue = df_clean['Revenue'].sum()
total_orders = df_clean['InvoiceNo'].nunique()
total_customers = df_clean['CustomerID'].nunique()
total_products = df_clean['StockCode'].nunique()
avg_order_value = total_revenue / total_orders
avg_revenue_per_cust = total_revenue / total_customers

metrics_df = pd.DataFrame({
    'Metric': [
        'Total Revenue (£)', 'Total Valid Orders', 'Total Unique Customers',
        'Total Unique Products', 'Average Order Value (£)', 'Average Customer Value (£)'
    ],
    'Value': [
        f"£{total_revenue:,.2f}", f"{total_orders:,}", f"{total_customers:,}",
        f"{total_products:,}", f"£{avg_order_value:,.2f}", f"£{avg_revenue_per_cust:,.2f}"
    ]
})
display(metrics_df)


In [ ]:
# 1. Monthly Revenue Trend
df_clean['InvoiceYearMonth'] = df_clean['InvoiceDate'].dt.to_period('M').astype(str)
monthly_rev = df_clean.groupby('InvoiceYearMonth')['Revenue'].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=monthly_rev, x='InvoiceYearMonth', y='Revenue', marker='o', color='#1f77b4', linewidth=2.5, ax=ax)
ax.set_title("Monthly Revenue Trend (2010 - 2011)", fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel("Year-Month", fontsize=11)
ax.set_ylabel("Revenue (£)", fontsize=11)
plt.xticks(rotation=45)
ax.yaxis.set_major_formatter('£{x:,.0f}')
plt.tight_layout()
plt.show()


In [ ]:
# 2. Top-Selling Products by Quantity and Revenue
top_qty = df_clean.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
top_rev = df_clean.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=top_qty.values, y=top_qty.index, palette='Blues_r', ax=ax1)
ax1.set_title("Top 10 Products by Quantity Sold", fontsize=13, fontweight='bold')
ax1.set_xlabel("Units Sold")
ax1.set_ylabel("")

sns.barplot(x=top_rev.values, y=top_rev.index, palette='Greens_r', ax=ax2)
ax2.set_title("Top 10 Products by Total Revenue (£)", fontsize=13, fontweight='bold')
ax2.set_xlabel("Revenue (£)")
ax2.set_ylabel("")
ax2.xaxis.set_major_formatter('£{x:,.0f}')

plt.tight_layout()
plt.show()


In [ ]:
# 3. Country-wise Revenue Distribution
country_rev = df_clean.groupby('Country').agg(
    Revenue=('Revenue', 'sum'),
    Orders=('InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique')
).sort_values('Revenue', ascending=False)

country_rev['Share (%)'] = (country_rev['Revenue'] / total_revenue) * 100
display(country_rev.head(10))

plt.figure(figsize=(12, 5))
top_countries = country_rev.head(8).reset_index()
sns.barplot(data=top_countries, x='Country', y='Revenue', palette='crest')
plt.title("Revenue by Top 8 Countries", fontsize=13, fontweight='bold')
plt.xticks(rotation=30)
plt.ylabel("Revenue (£)")
plt.gca().yaxis.set_major_formatter('£{x:,.0f}')
plt.tight_layout()
plt.show()


## Step 5: K-Means Customer Segmentation (RFM Analysis)

### Customer Features:
* **Recency ($R$):** Days elapsed between customer's most recent purchase and our consistent snapshot date ($T_{\text{ref}} = \max(\text{InvoiceDate}) + 1\text{ day}$).
* **Frequency ($F$):** Total count of unique orders (`InvoiceNo`) placed by the customer.
* **Monetary ($M$):** Total sum of spending (£) across all orders.
* **Average Order Value ($AOV$):** Mean spend per invoice ($M / F$).

### Mathematical Justification for Log Transformation:
RFM features exhibit strong positive right-skewness spanning multiple orders of magnitude. Because K-Means relies on Euclidean distance:
$$d(x, c) = \sqrt{\sum_{j=1}^p (x_j - c_j)^2}$$
Outlier values in Monetary would unilaterally dominate clustering geometry. Applying $\log(1 + x)$ normalizes variance, followed by `StandardScaler` ($\mu=0, \sigma=1$).


In [ ]:
# Step 5.1: Create Customer-Level RFM Dataset
ref_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

cust_df = df_clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda d: (ref_date - d.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum'),
    TotalItems=('Quantity', 'sum'),
    UniqueProducts=('StockCode', 'nunique'),
    FirstPurchase=('InvoiceDate', 'min'),
    LastPurchase=('InvoiceDate', 'max')
).reset_index()

cust_df['AverageOrderValue'] = cust_df['Monetary'] / cust_df['Frequency']
cust_df['AvgBasketSize'] = cust_df['TotalItems'] / cust_df['Frequency']
cust_df['PurchaseSpan'] = (cust_df['LastPurchase'] - cust_df['FirstPurchase']).dt.days

print("Customer-Level Feature Sample:")
display(cust_df[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'AverageOrderValue']].head())


In [ ]:
# Step 5.2: Transformation & Scaling
cluster_features = ['Recency', 'Frequency', 'Monetary', 'AverageOrderValue']

# Log transform skewed features
X_cluster = np.log1p(cust_df[cluster_features])

# Standardize to zero mean, unit variance
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

print("Scaled feature array shape:", X_cluster_scaled.shape)


In [ ]:
# Step 5.3: Determine Optimal Number of Clusters (Elbow & Silhouette)
k_range = range(2, 9)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_cluster_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Elbow plot
ax1.plot(k_range, inertias, marker='o', color='#1f77b4', linewidth=2)
ax1.set_title("Elbow Method (Inertia vs. K)", fontsize=13, fontweight='bold')
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Inertia / Within-Cluster Sum of Squares")

# Silhouette plot
ax2.plot(k_range, sil_scores, marker='s', color='#2ca02c', linewidth=2)
ax2.set_title("Silhouette Scores vs. K", fontsize=13, fontweight='bold')
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()

print(f"Optimal K selection: K=4 provides balanced separation and intuitive business interpretability.")


In [ ]:
# Step 5.4: Train Final K-Means Model (K=4)
K_OPTIMAL = 4
kmeans_model = KMeans(n_clusters=K_OPTIMAL, init='k-means++', n_init=10, random_state=RANDOM_STATE)
cust_df['Cluster'] = kmeans_model.fit_predict(X_cluster_scaled)

sil_val = silhouette_score(X_cluster_scaled, cust_df['Cluster'])
print(f"Fitted K-Means Model (k={K_OPTIMAL}) | Silhouette Score: {sil_val:.4f}")

# Profile clusters by their raw feature medians/means
profile_summary = cust_df.groupby('Cluster').agg(
    CustomerCount=('CustomerID', 'count'),
    AvgRecency=('Recency', 'mean'),
    AvgFrequency=('Frequency', 'mean'),
    AvgMonetary=('Monetary', 'mean'),
    AvgAOV=('AverageOrderValue', 'mean')
).reset_index()

# Sort clusters by monetary spending to assign meaningful business personas
monetary_order = profile_summary.sort_values('AvgMonetary')['Cluster'].tolist()
persona_map = {
    monetary_order[3]: "High-Value Customers",
    monetary_order[2]: "Frequent Customers",
    monetary_order[1]: "Potential / Budget Customers",
    monetary_order[0]: "At-Risk / Inactive Customers"
}

cust_df['Segment'] = cust_df['Cluster'].map(persona_map)
profile_summary['Segment'] = profile_summary['Cluster'].map(persona_map)

print("\n=== Customer Cluster Profiles & Business Personas ===")
display(profile_summary[['Cluster', 'Segment', 'CustomerCount', 'AvgRecency', 'AvgFrequency', 'AvgMonetary', 'AvgAOV']])


In [ ]:
# Step 5.5: Interactive Plotly Cluster Visualization & Model Serialization
fig_cluster = px.scatter(
    cust_df,
    x='Recency',
    y='Monetary',
    size='Frequency',
    color='Segment',
    log_y=True,
    hover_data=['CustomerID', 'AverageOrderValue'],
    title="Customer Segments: Recency vs. Monetary (Size = Frequency)",
    color_discrete_sequence=['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728'],
    labels={'Recency': 'Recency (Days since last order)', 'Monetary': 'Total Monetary Spend (£)'}
)
fig_cluster.show()

# Save models to models directory
models_dir = os.path.join("..", "models") if os.path.exists(os.path.join("..", "models")) else (
    "models" if os.path.exists("models") else os.path.join("..", "..", "models")
)
os.makedirs(models_dir, exist_ok=True)

joblib.dump(kmeans_model, os.path.join(models_dir, "kmeans_model.pkl"))
joblib.dump(scaler, os.path.join(models_dir, "scaler.pkl"))

cluster_meta = {
    'features': cluster_features,
    'log_transform': True,
    'n_clusters': K_OPTIMAL,
    'persona_map': persona_map,
    'silhouette_score': float(sil_val)
}
joblib.dump(cluster_meta, os.path.join(models_dir, "kmeans_metadata.pkl"))

# Export segmentation results
cust_df.to_csv(os.path.join(models_dir, "..", "dataset", "customer_segments.csv"), index=False)
print("Saved kmeans_model.pkl, scaler.pkl, kmeans_metadata.pkl, and exported customer_segments.csv.")


## Step 6: Market Basket Analysis with Apriori Algorithm

### Market Basket Metrics:
* **Support:** Proportion of transactions containing itemset $A$:
  $$\text{Support}(A) = \frac{\text{Transactions containing } A}{\text{Total Transactions}}$$
* **Confidence:** Probability of purchasing $B$ given $A$ was purchased:
  $$\text{Confidence}(A \to B) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)}$$
* **Lift:** Strength of association compared to random independence:
  $$\text{Lift}(A \to B) = \frac{\text{Confidence}(A \to B)}{\text{Support}(B)}$$
  * $\text{Lift} > 1$: Positive association (items co-occur more frequently than expected by chance).
  * $\text{Lift} = 1$: Independence.
  * $\text{Lift} < 1$: Negative association / substitution.

> **Important Note:** Association rules discover empirical co-occurrence patterns in historical basket data. They do **not** establish causation or guarantee individual purchase intent.


In [ ]:
# Step 6.1: Build Transaction Basket Matrix
item_lookup = df_clean.drop_duplicates(subset=['StockCode']).set_index('StockCode')['Description'].to_dict()

# Focus basket analysis on UK transactions (largest market with homogeneous purchasing behavior)
df_uk = df_clean[df_clean['Country'] == 'United Kingdom']

# Select top recurring items to ensure dense, meaningful associations and optimal memory usage
top_stock_codes = df_uk['StockCode'].value_counts().head(300).index
df_basket_input = df_uk[df_uk['StockCode'].isin(top_stock_codes)]

basket = df_basket_input.groupby(['InvoiceNo', 'StockCode'])['Quantity'].sum().unstack().fillna(0)
# Convert quantities to binary indicators (True if item present in basket, False otherwise)
basket_encoded = (basket > 0).astype(bool)

print(f"One-Hot Encoded Basket Shape: {basket_encoded.shape[0]:,} orders x {basket_encoded.shape[1]} product items")


In [ ]:
# Step 6.2: Frequent Itemsets & Association Rule Mining
MIN_SUPPORT = 0.02
MIN_LIFT = 1.0

# Generate frequent itemsets using Apriori
frequent_items = apriori(basket_encoded, min_support=MIN_SUPPORT, use_colnames=True)
print(f"Discovered {len(frequent_items)} frequent itemsets with min_support >= {MIN_SUPPORT}\n")

if len(frequent_items) > 0:
    # Generate association rules
    rules = association_rules(frequent_items, metric="lift", min_threshold=MIN_LIFT)
    rules['antecedents_desc'] = rules['antecedents'].apply(lambda x: [item_lookup.get(code, code) for code in x])
    rules['consequents_desc'] = rules['consequents'].apply(lambda x: [item_lookup.get(code, code) for code in x])
    
    # Sort top rules by Lift
    top_rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)
    
    print(f"Generated {len(top_rules)} valid association rules (min_lift >= {MIN_LIFT}).")
    print("\n--- Top 10 Product Association Rules (by Lift) ---")
    display(top_rules[['antecedents_desc', 'consequents_desc', 'support', 'confidence', 'lift']].head(10))
    
    # Bundle suggestions visualization
    plt.figure(figsize=(9, 5))
    scatter = plt.scatter(rules['support'], rules['confidence'], c=rules['lift'], cmap='viridis', alpha=0.7, s=50)
    cbar = plt.colorbar(scatter)
    cbar.set_label('Lift', rotation=270, labelpad=15)
    plt.title("Association Rules: Support vs. Confidence (Colored by Lift)", fontsize=13, fontweight='bold')
    plt.xlabel("Support")
    plt.ylabel("Confidence")
    plt.tight_layout()
    plt.show()
else:
    print("No frequent itemsets found for the configured support threshold.")


## Step 7: Random Forest Customer Spending Classification

### Preventing Data Leakage (Strict Isolation):
* **Target Variable:** Customer spending category classified into:
  1. `Low Spending` ($M \le Q_{33}$)
  2. `Medium Spending` ($Q_{33} < M \le Q_{67}$)
  3. `High Spending` ($M > Q_{67}$)
* **Zero Leakage Rule:** Neither total `Monetary` spend nor `AverageOrderValue` (since $\text{AOV} = M/F$) may be included as features. 
* **Allowed Predictive Behavioral Features:**
  1. `Recency` (Days since most recent order)
  2. `Frequency` (Number of orders placed)
  3. `TotalItems` (Total physical volume of items purchased)
  4. `UniqueProducts` (Product diversity purchased)
  5. `AvgBasketSize` (Average unit volume per order)
  6. `PurchaseSpan` (Days between first and last purchase)


In [ ]:
# Step 7.1: Construct Non-Leaking Classification Dataset
# Establish spending category boundaries
low_cutoff = cust_df['Monetary'].quantile(0.33)
high_cutoff = cust_df['Monetary'].quantile(0.67)

def assign_tier(spend):
    if spend <= low_cutoff:
        return 'Low Spending'
    elif spend <= high_cutoff:
        return 'Medium Spending'
    else:
        return 'High Spending'

cust_df['SpendingCategory'] = cust_df['Monetary'].apply(assign_tier)

print("Target Class Distribution:")
print(cust_df['SpendingCategory'].value_counts())
print(f"\nDocumented Thresholds: Low <= £{low_cutoff:.2f} | High > £{high_cutoff:.2f}")

# Select strictly non-leaking behavioral features
clf_feature_cols = ['Recency', 'Frequency', 'TotalItems', 'UniqueProducts', 'AvgBasketSize', 'PurchaseSpan']
X = cust_df[clf_feature_cols]
y = cust_df['SpendingCategory']

# Verify absence of Monetary or AOV features
assert 'Monetary' not in X.columns, "Data Leakage Error: Monetary feature present!"
assert 'AverageOrderValue' not in X.columns, "Data Leakage Error: AOV feature present!"
print("Feature set validated: Zero data leakage confirmed.")


In [ ]:
# Step 7.2: Stratified Train-Test Split & Model Training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=5,
    random_state=RANDOM_STATE
)
rf_classifier.fit(X_train, y_train)
y_pred = rf_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Out-of-Sample Accuracy: {accuracy * 100:.2f}%\n")
print("=== Detailed Classification Report ===")
print(classification_report(y_test, y_pred))


In [ ]:
# Step 7.3: Confusion Matrix & Feature Importances
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Confusion Matrix
classes = ['High Spending', 'Medium Spending', 'Low Spending']
cm = confusion_matrix(y_test, y_pred, labels=classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax1)
ax1.set_title("Random Forest Confusion Matrix", fontsize=13, fontweight='bold')
ax1.set_xlabel("Predicted Spending Category")
ax1.set_ylabel("True Spending Category")

# Feature Importance
importances = pd.Series(rf_classifier.feature_importances_, index=clf_feature_cols).sort_values(ascending=True)
importances.plot(kind='barh', color='#2ca02c', ax=ax2)
ax2.set_title("Feature Importances", fontsize=13, fontweight='bold')
ax2.set_xlabel("Gini Importance Score")

plt.tight_layout()
plt.show()


In [ ]:
# Step 7.4: Model Serialization & Single-Customer Inference Test
joblib.dump(rf_classifier, os.path.join(models_dir, "classification_model.pkl"))

clf_meta = {
    'features': clf_feature_cols,
    'thresholds': {'Low_Spending_Threshold': float(low_cutoff), 'High_Spending_Threshold': float(high_cutoff)},
    'classes': list(rf_classifier.classes_),
    'accuracy': float(accuracy),
    'feature_importances': importances.to_dict()
}
joblib.dump(clf_meta, os.path.join(models_dir, "classification_metadata.pkl"))
print("Saved models/classification_model.pkl and classification_metadata.pkl")

# Demonstration of single customer inference
sample_customer = pd.DataFrame([{
    'Recency': 15,          # Purchased 15 days ago
    'Frequency': 8,         # 8 orders placed
    'TotalItems': 450,      # 450 items
    'UniqueProducts': 65,   # 65 unique items
    'AvgBasketSize': 56.2,  # ~56 items per order
    'PurchaseSpan': 180     # Active over 180 days
}])

predicted_tier = rf_classifier.predict(sample_customer)[0]
probs = rf_classifier.predict_proba(sample_customer)[0]

print(f"\nInference Test:")
print(f"Predicted Spending Tier: >>> {predicted_tier} <<<")
for cls, prob in zip(rf_classifier.classes_, probs):
    print(f" - Probability {cls}: {prob*100:.1f}%")

print("\nNote: This model classifies customer spending tier using historical behavioral patterns. It does not forecast future lifetime value or customer churn.")
